# LAB-17 · Time a kernel honestly

Timing a GPU launch is like timing a delivery by watching the order leave your desk. You will warm up a real vector-add kernel, wait for completion, collect many samples and report their median and slow tail instead of trusting one stopwatch reading.

~20 minutes · **GPU optional**. A kernel is a small function run over many data items; Triton is a Python-based language for writing it. Run cells in order, fill the `# TODO` gaps, then use the checks; starter failures are expected.

Use a fresh Python runtime. Setup can install Triton in your account; it needs no API key or repository files. CPU correctness uses Triton's interpreter on a supported Linux runtime (Colab/Kaggle); native macOS is not a supported Triton runtime.


In [ ]:
import os
import sys
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
# The interpreter setting must be chosen BEFORE the first import of Triton.
if "triton" in sys.modules:
    raise RuntimeError("Restart the runtime, then run this setup cell first (Triton is already imported).")
if HAS_GPU:
    os.environ.pop("TRITON_INTERPRET", None)
else:
    os.environ["TRITON_INTERPRET"] = "1"
    print("No GPU: correctness checks only, timings skipped")
try:
    import triton
except ImportError:
    # Runs only in your notebook runtime; PyTorch is preinstalled in Colab/Kaggle.
    %pip install -q triton
    import triton
import triton.language as tl

GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU interpreter)"
print(f"torch {torch.__version__}; triton {triton.__version__}; GPU: {GPU_NAME}")
torch.manual_seed(0)


In [ ]:
# Self-check failures are reported without stopping Run all, so you can reach Solution.
# Solution checks are strict: a broken reference stops execution before benchmarking.
def check(label, test, strict=False):
    try:
        test()
    except Exception as error:
        print(f"FAIL {label}: {type(error).__name__}: {error}")
        if strict:
            raise
        return False
    print(f"PASS {label}")
    return True

# Vendor datasheet peaks, decimal GB/s. Match variants conservatively.
PEAK_GB_S = {"T4": 320.0, "P100 16 GB": 732.0, "H100 SXM": 3350.0}
def detected_peak():
    if not HAS_GPU:
        return None
    name = GPU_NAME.upper()
    if name in ("TESLA T4", "NVIDIA T4", "T4"):
        return PEAK_GB_S["T4"]
    memory = torch.cuda.get_device_properties(0).total_memory
    if "P100" in name and 15e9 <= memory <= 18e9:
        return PEAK_GB_S["P100 16 GB"]
    if "H100" in name and "SXM" in name:
        return PEAK_GB_S["H100 SXM"]
    return None

def report(label, milliseconds, byte_count):
    gb_s = byte_count / (milliseconds * 1e6)
    text = f"your run | {label}: {milliseconds:.4f} ms, {byte_count:,} logical bytes, {gb_s:.2f} GB/s"
    peak = detected_peak()
    if peak is not None:
        text += f", {100 * gb_s / peak:.1f}% of vendor peak"
    print(text)

def bench(label, call, byte_count):
    if not HAS_GPU:
        print(f"Timings skipped: {label} (CPU correctness only)")
        return
    from triton.testing import do_bench
    milliseconds = do_bench(call, warmup=25, rep=100)
    report(label, milliseconds, byte_count)


## A known workload

This notebook includes its own vector-add kernel so it needs no other lab files. A mask guards the last partial block, like skipping empty seats in a theater. First check the values, before drawing conclusions from speed.


In [ ]:
@triton.jit
def add_kernel(A, B, Out, N: tl.constexpr, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    a = tl.load(A + offsets, mask=mask, other=0)
    b = tl.load(B + offsets, mask=mask, other=0)
    value = a + b
    tl.store(Out + offsets, value, mask=mask)

def vector_add(a, b, out=None):
    assert a.ndim == b.ndim == 1 and a.shape == b.shape
    assert a.is_contiguous() and b.is_contiguous()
    if out is None:
        out = torch.empty_like(a)
    if a.numel():
        add_kernel[(triton.cdiv(a.numel(), 256),)](a, b, out, a.numel(), 256)
    return out


In [ ]:
def check_add():
    # Illustrative sizes: empty, singleton, partial block and multiple programs.
    for n in (0, 1, 255, 256, 1000):
        a = torch.randn(n, device=DEVICE, dtype=torch.float32)
        b = torch.randn_like(a)
        torch.testing.assert_close(vector_add(a, b), a + b, rtol=1e-5, atol=1e-6)
check("vector add, including a masked tail", check_add)


## 1 · Wait for the delivery

A CUDA event is a timestamp recorded on the GPU’s work stream. Fill in the harness: launch several warm-ups, call `torch.cuda.synchronize()` to wait for outstanding work, then collect one elapsed-millisecond event sample per repeat, synchronizing before reading it. Use the current stream; outputs are preallocated and compilation belongs in warm-up, outside the samples.


In [ ]:
import math
import time

def measure(call, warmup=10, repeats=100):
    # TODO: warm up, synchronize, record CUDA events for each repeat, synchronize, collect ms.
    raise NotImplementedError("Finish the warm-up / synchronization / repeats harness")


In [ ]:
def check_measure():
    if not HAS_GPU:
        # No fake GPU timings: validate the real kernel on CPU and defer the CUDA harness.
        a = torch.arange(257, device=DEVICE, dtype=torch.float32)
        torch.testing.assert_close(vector_add(a, a), a + a, rtol=1e-5, atol=1e-6)
        print("CUDA warm-up/event/synchronization checks skipped: GPU required")
        return
    a = torch.arange(257, device=DEVICE, dtype=torch.float32)
    b = torch.ones_like(a)
    out = torch.empty_like(a)
    calls = []
    def call():
        calls.append(1)
        return vector_add(a, b, out)
    samples = measure(call, warmup=3, repeats=20)
    torch.testing.assert_close(out, a + b, rtol=1e-5, atol=1e-6)
    torch.testing.assert_close(torch.tensor(len(calls)), torch.tensor(23), rtol=0, atol=0)
    assert samples.shape == (20,) and torch.isfinite(samples).all() and (samples > 0).all(), "Return one positive ms sample per repeat"
check("kernel correctness / CUDA harness when available", check_measure)


## 2 · Report the middle and the slow tail

The median (p50) is the middle of the sorted samples; p90 and p99 sit near the slow end, like the commute times on unusually bad days. Complete the interpolation so a percentile between two samples uses a proportional part of each. Small sample counts make tail estimates noisy; these are local kernel latency percentiles, not server request latency or goodput.


In [ ]:
def summarize(samples):
    # Linear interpolation between sorted neighbors (same convention as torch.quantile).
    ordered = sorted(float(x) for x in samples)
    if len(ordered) < 2 or any(not math.isfinite(x) or x <= 0 for x in ordered):
        raise ValueError("Need at least two finite positive times.")
    def percentile(q):
        position = (len(ordered) - 1) * q
        low, high = math.floor(position), math.ceil(position)
        return ordered[low]  # TODO: interpolate toward the high neighbor
    return torch.tensor([percentile(.5), percentile(.9), percentile(.99)], dtype=torch.float64)


In [ ]:
def check_summary():
    # Illustrative distribution, NOT GPU measurements; also tests unsorted/tied samples.
    for samples in ([9, 1, 4, 2, 7, 3], [1, 1, 1, 1], [1, 100]):
        values = torch.tensor(samples, dtype=torch.float64)
        expected = torch.quantile(values, torch.tensor([.5, .9, .99], dtype=torch.float64))
        torch.testing.assert_close(summarize(samples), expected, rtol=1e-12, atol=1e-12)
check("median, p90 and p99 interpolation", check_summary)


## Try first

Try filling the TODOs and rerunning each exercise and its check before reading on. Run all intentionally reports starter failures, then installs the complete definitions below and checks them again. Only checked solutions are timed.


## Solution


In [ ]:
def measure(call, warmup=10, repeats=100):
    if not HAS_GPU:
        raise RuntimeError("GPU timing requires a GPU; CPU correctness checks are separate.")
    if warmup < 1 or repeats < 2:
        raise ValueError("Use warm-up and multiple repeats.")
    for _ in range(warmup):
        call()
    torch.cuda.synchronize()
    samples = []
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    for _ in range(repeats):
        start.record()
        call()
        end.record()
        torch.cuda.synchronize()
        samples.append(start.elapsed_time(end))
    return torch.tensor(samples, dtype=torch.float64)


In [ ]:
def summarize(samples):
    # Linear interpolation between sorted neighbors (same convention as torch.quantile).
    ordered = sorted(float(x) for x in samples)
    if len(ordered) < 2 or any(not math.isfinite(x) or x <= 0 for x in ordered):
        raise ValueError("Need at least two finite positive times.")
    def percentile(q):
        position = (len(ordered) - 1) * q
        low, high = math.floor(position), math.ceil(position)
        return ordered[low] + (ordered[high] - ordered[low]) * (position - low)
    return torch.tensor([percentile(.5), percentile(.9), percentile(.99)], dtype=torch.float64)


In [ ]:
check("solution: kernel / CUDA harness", check_measure, strict=True)
check("solution: percentiles", check_summary, strict=True)


## Measure on a GPU

Results below are **your run**, not a promised speedup. Byte counts are logical tensor reads/writes, excluding allocation, cache effects and instruction overhead; GB/s is effective bandwidth, not measured HBM traffic, and can exceed a datasheet peak when data is cached. Times depend on shape, GPU, compiler and runtime.

The small peak table uses vendor datasheets: T4 320 GB/s, P100 **16 GB** 732 GB/s, H100 **SXM** 3.35 TB/s. Ambiguous or unknown GPU variants show GB/s without a peak percentage. No timings run in CPU-interpreter mode.


## The asynchronous stopwatch trap

The first host stopwatch below ends immediately after launches; it omits the final wait. A second host time includes completion of the same batch. CPU submission overhead can dominate tiny kernels, so the unsynchronized host number is not guaranteed to be smaller than the CUDA-event time on every run; neither host number is a pure kernel latency. CUDA event samples then report GPU stream time (including any gaps between the events), with median and tail spread.

This is an isolated microbenchmark, not a load test: I36/I37 also need per-request arrival, first-token and completion timestamps, plus explicit latency targets to measure serving goodput.


In [ ]:
if HAS_GPU:
    n = 2**22  # Illustrative workload; 100 measurements, not a claimed timing.
    a = torch.randn(n, device=DEVICE)
    b = torch.randn_like(a)
    out = torch.empty_like(a)
    call = lambda: vector_add(a, b, out)
    for _ in range(10):
        call()
    torch.cuda.synchronize()
    repeats = 100
    host_start = time.perf_counter()
    for _ in range(repeats):
        call()
    submission_ms = (time.perf_counter() - host_start) * 1000 / repeats
    torch.cuda.synchronize()
    completed_ms = (time.perf_counter() - host_start) * 1000 / repeats
    print(f"your run | misleading no-final-sync host time: {submission_ms:.4f} ms/launch")
    print(f"your run | completed host batch time: {completed_ms:.4f} ms/launch")
    samples = measure(call, warmup=10, repeats=repeats)
    median, p90, p99 = summarize(samples).tolist()
    torch.testing.assert_close(out, a + b, rtol=1e-5, atol=1e-6)
    print(f"your run | {repeats} CUDA event samples: median {median:.4f} ms, p90 {p90:.4f} ms, p99 {p99:.4f} ms")
    report("vector add (median CUDA event time)", median, 3 * n * a.element_size())
else:
    print("Timings skipped (CPU correctness only); percentile fixture checks above still run")


## Take it further

You built a warm-up, synchronization and repeated-sample harness with median/p90/p99 reporting. Real performance studies also vary shapes, report software/hardware settings and control other GPU activity; serving studies add request latency, throughput and goodput. Backs I35–I37 (measurement and metrics); kernel percentiles alone do not measure an LLM server.

[Academy LAB-17 page](https://tile-and-error-academy.vercel.app/#lab/lab-17) · [Tile & Error channel](https://www.youtube.com/@tileanderror)
